# SkinFLNet++ — experiments (local Jupyter or Google Colab)

Run sections **in order** the first time on Colab: **setup → install → download ISIC → manifests → imports →** individual experiments.

**Runtime**

- **Local (VS Code, Cursor, Jupyter):** open this notebook from the repo or from `notebooks/`. Setup auto-detects the repo root (`pyproject.toml` + `src/`). Optional: set `PROJECT_ROOT_OVERRIDE` in the setup cell, or `SKINFL_RUNTIME=local` in the environment.
- **Colab + Drive:** sync the repo to `MyDrive/SkinFL` (or change `COLAB_DRIVE_ROOT`). The setup cell mounts Drive so **code**, **configs**, **results**, and **figures** live there. **`DATA_ROOT`** is **`COLAB_SESSION_DATA_ROOT`** (default `/content/skinfl_data`) so ISIC stays on fast VM-local disk, not Drive. Override with `SKINFL_RUNTIME=local` inside Colab only if you need paths relative to this runtime.

**Before you start**

- **GPU** helps for FL (Runtime → GPU on Colab).
- **Python 3.11+** (`pyproject.toml`).
- **ISIC downloads:** archives are tens of gigabytes Colab ephemeral disk fills and **new VMs require re-download**; session `DATA_ROOT` is faster than Drive for training I/O but not persistent across disconnects unless you checkpoint elsewhere.
- **`num_workers`:** if DataLoader workers misbehave, use `0` or `2` in the YAML.
- **Manifest `client_id`:** updated by each experiment run when partitioning changes (`RUN_EXPERIMENTS.md`).
- **Manifests:** the manifest cell builds **ISIC2019** after the download cell fills `DATA_ROOT/ISIC2019/`.
- **`SKINFL_DROP_MISSING_IMAGES`:** with `colab_drive`, missing rows are skipped by default (legacy partial-upload behavior). **`DROP_MISSING_IMAGES = False`** is recommended once the official ISIC download cell has extracted all training JPEGs (`strict`/paper-aligned counts). Locally, nothing is set unless you export the variable yourself.


In [ ]:
from __future__ import annotations

import logging
import os
import sys
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError(
        f"This project requires Python >= 3.11 (pyproject.toml). Got: {sys.version}"
    )


def _in_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except ImportError:
        return False


def _runtime_mode() -> str:
    """local = this machine; colab_drive = mount Drive and use COLAB_DRIVE_ROOT."""
    env = os.environ.get("SKINFL_RUNTIME", "").strip().lower()
    if env in ("local", "colab_drive"):
        return env
    return "colab_drive" if _in_colab() else "local"


# Editable: where the repo lives on Google Drive (must contain pyproject.toml, scripts/, …)
COLAB_DRIVE_ROOT = Path("/content/drive/MyDrive/SkinFL")

# VM-local datasets on Colab (fast I/O); never under Drive — see setup + download cells.
COLAB_SESSION_DATA_ROOT = Path("/content/skinfl_data")

# Local: set to a Path to pin the repo root, or None to search from cwd / notebooks/
PROJECT_ROOT_OVERRIDE: Path | None = None

# Drive/Colab: True skips manifest rows whose image file is missing (typical for partial uploads).
# False = strict: require every training image path. Recommended after a full official ISIC session download.
# None = auto: True on colab_drive only (backward-compatible with partial uploads).
DROP_MISSING_IMAGES: bool | None = None


def _discover_project_root() -> Path:
    start = Path.cwd().resolve()
    candidates = [start]
    if start.name == "notebooks":
        candidates.append(start.parent)
    for cand in candidates:
        if (cand / "pyproject.toml").is_file() and (cand / "src").is_dir():
            return cand
    for parent in start.parents:
        if (parent / "pyproject.toml").is_file() and (parent / "src").is_dir():
            return parent
    raise FileNotFoundError(
        "Could not find SkinFL repo (pyproject.toml + src/). "
        "cd to repo root or notebooks/, or set PROJECT_ROOT_OVERRIDE, "
        "or SKINFL_RUNTIME=colab_drive on Colab."
    )


RUNTIME = _runtime_mode()

if RUNTIME == "colab_drive":
    try:
        from google.colab import drive
    except ImportError as e:
        raise SystemExit(
            'SKINFL_RUNTIME=colab_drive but not in Colab. '
            "Unset SKINFL_RUNTIME or use local Jupyter."
        ) from e
    drive.mount("/content/drive")
    PROJECT_ROOT = COLAB_DRIVE_ROOT
else:
    PROJECT_ROOT = PROJECT_ROOT_OVERRIDE or _discover_project_root()

if RUNTIME == "colab_drive":
    DATA_ROOT = COLAB_SESSION_DATA_ROOT
else:
    DATA_ROOT = PROJECT_ROOT / "data"

# ISIC 2019 runs live under results/isic2019/ and figures/isic2019/ (separate from 2018)
RESULTS_DIR = PROJECT_ROOT / "results" / "isic2019"
FIGURES_DIR = PROJECT_ROOT / "figures" / "isic2019"

DATA_ROOT.mkdir(parents=True, exist_ok=True)
for path in (RESULTS_DIR, FIGURES_DIR):
    path.mkdir(parents=True, exist_ok=True)

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError(
        f"Invalid PROJECT_ROOT: {PROJECT_ROOT} (pyproject.toml missing)."
    )

os.chdir(PROJECT_ROOT)

_repo_root = str(PROJECT_ROOT.resolve())
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

_drop = (RUNTIME == "colab_drive") if DROP_MISSING_IMAGES is None else DROP_MISSING_IMAGES
if _drop:
    os.environ["SKINFL_DROP_MISSING_IMAGES"] = "1"
else:
    os.environ.pop("SKINFL_DROP_MISSING_IMAGES", None)

print("RUNTIME:", RUNTIME)
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print(
    "SKINFL_DROP_MISSING_IMAGES:",
    os.environ.get("SKINFL_DROP_MISSING_IMAGES", "(unset — strict image checks)"),
)
if _drop:
    print(
        "Note: rows without a file on disk are dropped (subset run). "
        "Set DROP_MISSING_IMAGES=False after a full extract (official ISIC download cell) for strict counts."
    )


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUNTIME: colab_drive
PROJECT_ROOT: /content/drive/MyDrive/SkinFL
DATA_ROOT: /content/skinfl_data
RESULTS_DIR: /content/drive/MyDrive/SkinFL/results/isic2019
FIGURES_DIR: /content/drive/MyDrive/SkinFL/figures/isic2019
SKINFL_DROP_MISSING_IMAGES: 1
Note: rows without a file on disk are dropped (subset run). Set DROP_MISSING_IMAGES=False after a full extract (official ISIC download cell) for strict counts.


In [ ]:
%pip install -q -e ".[dev]"


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for skinflnet-plus (pyproject.toml) ... done


In [ ]:

# ISIC 2019: official challenge files -> DATA_ROOT/ISIC2019/ (Colab session disk; large downloads).

import shutil
import zipfile
from urllib.request import Request, urlopen

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None


def _download_file(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    req = Request(url, headers={"User-Agent": "SkinFL-ISIC-fetch/1.0"})
    with urlopen(req) as resp:  # noqa: S310 — curated ISIC S3 URLs only
        hdr = resp.headers.get("Content-Length")
        total = int(hdr) if hdr is not None and hdr.isdigit() else None
        bs = 256 * 1024

        if tqdm is None:
            with open(dest, "wb") as f:
                while True:
                    chunk = resp.read(bs)
                    if not chunk:
                        break
                    f.write(chunk)
            return

        with open(dest, "wb") as f, tqdm(
            desc=dest.name[:48],
            total=total,
            unit="iB",
            unit_scale=True,
            unit_divisor=1024,
            miniters=1,
        ) as bar:
            while True:
                chunk = resp.read(bs)
                if not chunk:
                    break
                f.write(chunk)
                bar.update(len(chunk))



urls = [
    "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip",
    "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_Input.zip",
    "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Metadata.csv",
    "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv",
    "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_Metadata.csv",
    "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_GroundTruth.csv",
]

# True = always re-fetch and re-extract (slow; frees nothing under ISIC2019 except overwritten files).
FORCE_ISIC_REDOWNLOAD = False


def _isic_training_ready() -> bool:
    root = DATA_ROOT / "ISIC2019"
    gt = root / "ISIC_2019_Training_GroundTruth.csv"
    meta = root / "ISIC_2019_Training_Metadata.csv"
    img_dir = root / "ISIC_2019_Training_Input"
    if not gt.is_file() or not meta.is_file() or not img_dir.is_dir():
        return False
    return any(img_dir.glob("*.jpg"))


if RUNTIME != "colab_drive":
    print(
        "SKIP: ISIC download is for Colab + Drive runtime only. "
        "Local runs use PROJECT_ROOT/data (no auto-download)."
    )
elif _isic_training_ready() and not FORCE_ISIC_REDOWNLOAD:
    print(
        "SKIP: ISIC2019 training CSVs + images already present under",
        DATA_ROOT / "ISIC2019",
    )
else:
    isic_dest = DATA_ROOT / "ISIC2019"
    staging = DATA_ROOT / ".cache_isic2019"
    isic_dest.mkdir(parents=True, exist_ok=True)
    staging.mkdir(parents=True, exist_ok=True)

    for url in urls:
        name = Path(url.rstrip("/")).name
        out = staging / name
        _download_file(url, out)

        if name.endswith(".csv"):
            shutil.copy2(out, isic_dest / name)
            print("  -> copied CSV to", isic_dest / name)
        elif name.endswith(".zip"):
            print("  -> extracting:", name, flush=True)
            with zipfile.ZipFile(out, "r") as zf:
                infos = zf.infolist()
                if tqdm is not None:
                    for m in tqdm(infos, desc=f"Unpack {name[:42]}", unit="file"):
                        zf.extract(m, isic_dest)
                else:
                    zf.extractall(isic_dest)

    train_in = isic_dest / "ISIC_2019_Training_Input"
    if not train_in.is_dir():
        raise FileNotFoundError(
            f"After extract, expected directory missing: {train_in} (check ZIP layout)."
        )

    # Free Colab disk: remove staging artifacts (CSVs and ZIPs already copied / extracted).
    for p in staging.iterdir():
        p.unlink(missing_ok=True)

    print("Done. Training layout:", train_in.resolve())


ISIC_2019_Training_Input.zip:   0%|          | 0.00/9.10G [00:00<?, ?iB/s]

  -> extracting: ISIC_2019_Training_Input.zip


Unpack ISIC_2019_Training_Input.zip:   0%|          | 0/25334 [00:00<?, ?file/s]

ISIC_2019_Test_Input.zip:   0%|          | 0.00/3.56G [00:00<?, ?iB/s]

  -> extracting: ISIC_2019_Test_Input.zip


...

  -> copied CSV to /content/skinfl_data/ISIC2019/ISIC_2019_Training_GroundTruth.csv


ISIC_2019_Test_Metadata.csv:   0%|          | 0.00/287k [00:00<?, ?iB/s]

  -> copied CSV to /content/skinfl_data/ISIC2019/ISIC_2019_Test_Metadata.csv


ISIC_2019_Test_GroundTruth.csv:   0%|          | 0.00/454k [00:00<?, ?iB/s]

  -> copied CSV to /content/skinfl_data/ISIC2019/ISIC_2019_Test_GroundTruth.csv
Done. Training layout: /content/skinfl_data/ISIC2019/ISIC_2019_Training_Input


In [ ]:
# Colab default: build ISIC2019 manifest only.
from src.data.manifest_build import ensure_manifest_for_dataset

_root = Path(DATA_ROOT)
ensure_manifest_for_dataset(_root, "isic2019")


In [ ]:
from src.fl.experiment_runner import run_experiment_from_yaml


def run_experiment(config_rel: str, **kwargs) -> None:
    """Run one YAML config in-process; writes under RESULTS_DIR and FIGURES_DIR.

    Extra kwargs match ``run_experiment_from_yaml`` (e.g. ``wandb_cli=True``,
    ``no_round_progress=True``, ``show_round_progress=False``).
    """
    path = Path(config_rel)
    if not path.is_absolute():
        path = PROJECT_ROOT / config_rel
    run_experiment_from_yaml(
        path,
        data_root=DATA_ROOT,
        results_dir=RESULTS_DIR,
        figures_dir=FIGURES_DIR,
        **kwargs,
    )


## Primary experiments

**Core ISIC2019 runs:** federated primary (`isic2019_dirichlet.yaml`) and pooled **centralized** upper bound (`centralized_upperbound.yaml`). Long GPU runtimes.


In [ ]:
run_experiment("configs/isic2019_dirichlet.yaml")  # Main FL benchmark — Dirichlet α=100


In [ ]:
run_experiment("configs/centralized_upperbound.yaml")  # Pooled ISIC2019, no FL


INFO | Experiment: centralized_upperbound
INFO | ISIC2019 manifest already exists at /content/skinfl_data/ISIC2019/manifest.csv (skipping)
INFO | IID partition: 10 clients, 20264 train images
INFO | Partition 'iid' applied (alpha=0.50, K=10, seed=42)
INFO | Partition figure saved: /content/drive/MyDrive/SkinFL/figures/isic2019/partition_isic2019_iid_K10_a0.5.png
INFO | Running CENTRALIZED upper-bound training
INFO | Centralized | 20264 train | 2533 val | 2534 test samples | best_checkpoint_on=test
WARNING | centralized_best_on=test: early stopping and best_model.pth use the TEST set — biased vs real deployment; use for FL-protocol parity only.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be ab

model.safetensors:   0%|          | 0.00/554M [00:00<?, ?B/s]

Streaming output truncated to the last 5000 lines.
Epoch 51/100: 100%|█████████▉| 633/634 [03:48<00:00,  2.53it/s]
Centralized | centralized_upperbound:  48%|████▊     | 45/94 [2:58:10<3:13:35, 237.05s/epoch] , tLoss=0.0806         
Epoch 52/100: 100%|█████████▉| 633/634 [03:46<00:00,  2.86it/s]
                                                               INFO | Epoch 52/100 (round 26) | TrainLoss=0.0802 | TestAcc=0.8303 | TestF1=0.7686
Centralized | centralized_upperbound:  49%|████▉     | 46/94 [3:02:17<3:10:14, 237.81s/epoch] , vAcc=0.830, vF1=0.769
Epoch 53/100: 100%|█████████▉| 633/634 [03:46<00:00,  2.78it/s]
Centralized | centralized_upperbound:  50%|█████     | 47/94 [3:06:03<3:05:35, 236.92s/epoch] , tLoss=0.0748         
Epoch 54/100: 100%|█████████▉| 633/634 [03:46<00:00,  2.71it/s]
                                                               INFO | Epoch 54/100 (round 27) | TrainLoss=0.0716 | TestAcc=0.8303 | TestF1=0.7867
Centralized | centralized_upperbound:  51%|████

## Partition ablations (ISIC2019, 10 clients)

**Dirichlet α sweep:** 0.1 (strong skew), 0.5, 1.0. The headline run above uses **α=100** separately. Optional IID/patient YAMLs remain in `configs/` if needed.


In [ ]:
run_experiment("configs/ablation_partition_dirichlet_01.yaml")  # α=0.1


In [ ]:
run_experiment("configs/ablation_partition_dirichlet_05.yaml")  # α=0.5


In [ ]:
run_experiment("configs/ablation_partition_dirichlet_10.yaml")  # α=1.0


In [ ]:
# Optional: IID or patient-proxy partition — uncomment if needed:
# run_experiment("configs/ablation_partition_iid.yaml")
# run_experiment("configs/ablation_partition_patient.yaml")


## Client-count ablations


In [ ]:
run_experiment("configs/ablation_nclients_5.yaml")  # 5 clients


In [ ]:
run_experiment("configs/ablation_nclients_10.yaml")  # 10 clients


In [ ]:
run_experiment("configs/ablation_nclients_20.yaml")  # 20 clients


## Strategy ablations


In [ ]:
run_experiment("configs/ablation_strategy_fedprox.yaml")


INFO | Experiment: ablation_strategy_fedprox
INFO | ISIC2019 manifest already exists at /content/skinfl_data/ISIC2019/manifest.csv (skipping)
INFO | Dirichlet(alpha=0.50) partition: 10 clients, sizes={0: 1805, 1: 1034, 2: 498, 3: 1295, 4: 7362, 5: 1458, 6: 1572, 7: 1195, 8: 2821, 9: 1224}
INFO | Partition 'dirichlet' applied (alpha=0.50, K=10, seed=42)
INFO | Partition figure saved: /content/drive/MyDrive/SkinFL/figures/isic2019/partition_isic2019_dirichlet_K10_a0.5.png
INFO | Running FEDERATED simulation
INFO | Starting custom simulation | Run: ablation_strategy_fedprox | Dataset: isic2019 | Backbone: vgg16_bn | Clients: 10 | Rounds: 50 | Strategy: fedprox | Partition: dirichlet(α=0.50)
INFO | Model: vgg16_bn | Classes: 8 | Params: 14,789,832 total (14,789,832 trainable) | Device: cuda
INFO | resume=True but no valid checkpoint; starting from scratch.
INFO | Model: vgg16_bn | Classes: 8 | Params: 14,789,832 total (14,789,832 trainable) | Device: cuda
INFO | Client 0 | train=1444 local

FL rounds | ablation_strategy_fedprox:   0%|          | 0/50 [00:00<?, ?round/s] 

INFO | Round 1/50 | sampled clients (5/10): 1, 6, 5, 7, 8
INFO |   Client 1 training ...
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
WARNING | AUROC/AUPRC computation failed: Number of classes in y_true not equal to the number of columns in 'y_score'
INFO |   Client 1 done | train_loss=1.3854 | n_train=827 | local_test_macro_f1=0.3132
INFO |   Client 6 training ...
WARNING | AUROC/AUPRC computation failed: Number of classes in y_true not equal to the number of columns in 'y_score'
INFO |   Client 6 done | train_loss=1.3500 | n_train=1257 | local_test_macro_f1=0.3380
INFO |   Client 5 training ...
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
WARNING | AUROC/AUPRC computation failed: Number of classes in

In [ ]:
run_experiment("configs/ablation_strategy_fedadam.yaml")


## Local-epoch ablations


In [ ]:
run_experiment("configs/ablation_localepochs_1.yaml")


In [ ]:
run_experiment("configs/ablation_localepochs_5.yaml")


In [ ]:
run_experiment("configs/ablation_localepochs_10.yaml")


## Optional: viz / debug config

Skip unless you use `configs/test_viz.yaml` intentionally.


In [ ]:
# run_experiment("configs/test_viz.yaml")


## Optional: reporting tables and figures

Writes aggregate markdown tables and extra curves/matrices into the same `--results-dir` / `--figures-dir` as experiments (`docs/reproducibility.md` helpers).


In [ ]:
from src.report.tables import write_report_tables

report_md = PROJECT_ROOT / "REPORT_TABLES_ISIC2019.md"
write_report_tables(RESULTS_DIR, report_md)
print("Wrote:", report_md)


In [ ]:
from src.report.figures import generate_report_figures

generate_report_figures(RESULTS_DIR, FIGURES_DIR, "isic2019")


In [ ]:
from src.report.figures import generate_report_figures
